## Chunk/test

In [4]:
import os
import json
import re

directory = "chunk/test"
mismatched_files = []

for filename in os.listdir(directory):
    if not filename.endswith(".json"):
        continue
    filepath = os.path.join(directory, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            continue
            
    ttbc_names = []
    for bc in data.get("THONG_TIN_CHUNG", {}).get("Thong_Tin_Bi_Cao", []):
        if "Ho_Ten" in bc:
            ttbc_names.append(bc["Ho_Ten"].strip())
            
    pq_names = []
    for pq in data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []):
        if "Bi_Cao" in pq:
            pq_names.append(pq["Bi_Cao"].strip())
            
    summary_names = []
    for summary in data.get("Synthetic_summary", []):
        match = re.search(r"tôi (?:tên )?là\s+([^,\.]+)", summary, re.IGNORECASE)
        if match:
            summary_names.append(match.group(1).strip())
            
    set_ttbc = set(ttbc_names)
    set_pq = set(pq_names)
    set_summary = set(summary_names)
    
    if set_ttbc != set_pq or set_ttbc != set_summary:
        mismatched_files.append({
            "file": filename,
            "Thong_Tin_Bi_Cao": ttbc_names,
            "PHAN_QUYET_CUA_TOA_SO_THAM": pq_names,
            "Synthetic_summary_extracted": summary_names
        })

print(f"Found {len(mismatched_files)} files with name mismatches:\n")
for item in mismatched_files:
    print(f"File: {item["file"]}")
    print(f"  Thong_Tin_Bi_Cao: {item["Thong_Tin_Bi_Cao"]}")
    print(f"  PHAN_QUYET_CUA_TOA_SO_THAM: {item["PHAN_QUYET_CUA_TOA_SO_THAM"]}")
    print(f"  Synthetic_summary (extracted): {item["Synthetic_summary_extracted"]}")
    print("-" * 60)


Found 0 files with name mismatches:



In [5]:
import shutil
import os

target_dir = "chunk/test2"
os.makedirs(target_dir, exist_ok=True)

moved_count = 0
for item in mismatched_files:
    filename = item["file"]
    source_path = os.path.join("chunk/test", filename)
    target_path = os.path.join(target_dir, filename)
    if os.path.exists(source_path):
        shutil.move(source_path, target_path)
        moved_count += 1
print(f"Moved {moved_count} files to {target_dir}")


Moved 0 files to chunk/test2
